# Hybrid Search using BM25 and Dense Retrieval in Chroma
---
This notebook implements a hybrid search system combining BM25 (lexical) and dense (semantic) retrieval methods. The goal is to improve search relevance by leveraging both keyword and semantic signals.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader

def load_pdf(file_path):
    loader = PyMuPDFLoader(file_path)
    docs = loader.load()
    return "\n".join([doc.page_content for doc in docs])

pdf_path = "rag_sample.pdf"
text = load_pdf(pdf_path)

/home/web-h-006/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import re
from langchain_text_splitters import MarkdownHeaderTextSplitter

def chunk_text(text):
    def convert_to_markdown(text):
        # SECTION level → H1
        text = re.sub(
            r'(SECTION\s+\d+:[^\n]+)',
            r'# \1',
            text
        )
        # 1.1 → H2
        text = re.sub(
            r'\n(\d+\.\d+\s+[A-Z][^\n]+)',
            r'\n## \1',
            text
        )
        # 1.1.1 → H3
        text = re.sub(
            r'\n(\d+\.\d+\.\d+\s+[A-Z][^\n]+)',
            r'\n### \1',
            text
        )
        return text

    headers_to_split_on = [
        ("#", "Section"),
        ("##", "Subsection"),
        ("###", "Sub-subsection"),
    ]
    splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    return splitter.split_text(convert_to_markdown(text))

docs = chunk_text(text)
print(len(docs))

48


In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [5]:
from langchain_chroma import Chroma

def create_vectorstore():
    vector_store = Chroma(
        collection_name="p1_hybrid_search",
        embedding_function=embeddings,
        host="localhost",       # Chroma server host
        port=8081               # exposed Docker port
    )
    vector_store.add_documents(documents=docs)
    return vector_store

def get_vectorstore(chunks):
    return Chroma.from_documents(
        documents=chunks,
        collection_name="p1_hybrid_search",
        embedding=embeddings,
    )

In [8]:
vector_store=get_vectorstore(chunks=docs)

In [10]:
def dense_retrieval(query, k=5, vector_store=vector_store):
    return vector_store.similarity_search(query=query, k=k)

d = dense_retrieval("operation paperclip")
d

[Document(id='73d4b74b-f40c-49d2-a207-1b30306ba923', metadata={'Sub-subsection': '1.1.1 OPERATION PAPERCLIP AND SOVIET ASSIMILATION', 'Subsection': '1.1 THE CAPTURE OF AEROSPACE TECHNOLOGY', 'Section': 'SECTION 1: POST-WORLD WAR II ROCKETRY FOUNDATIONS'}, page_content='The United States initiated Operation Paperclip, a strategic military program designed to\nrecruit prominent German scientists. Foremost among these was Wernher von Braun,\nwhose expertise in orbital mechanics and propulsion was unparalleled at the time. Von\nBraun and his team were relocated to Fort Bliss, Texas, and later to the Redstone Arsenal in\nHuntsville, Alabama. Simultaneously, the Soviet Union executed Operation Osoaviakhim,\nsecuring the services of numerous German technicians and vast quantities of V-2\ndocumentation. Sergei Korolev, who would later become the Chief Designer of the Soviet\nspace program, was tasked with reverse-engineering the V-2, resulting in the creation of the\nR-1 missile. The parallel 

In [11]:
from langchain_community.retrievers import BM25Retriever

def sparse_retrieval(query, k=5):
    retriever = BM25Retriever.from_documents(docs)
    return retriever.invoke(query)

s = sparse_retrieval("Operation Paperclip")
s

[Document(metadata={'Section': 'SECTION 1: POST-WORLD WAR II ROCKETRY FOUNDATIONS', 'Subsection': '1.1 THE CAPTURE OF AEROSPACE TECHNOLOGY', 'Sub-subsection': '1.1.1 OPERATION PAPERCLIP AND SOVIET ASSIMILATION'}, page_content='The United States initiated Operation Paperclip, a strategic military program designed to\nrecruit prominent German scientists. Foremost among these was Wernher von Braun,\nwhose expertise in orbital mechanics and propulsion was unparalleled at the time. Von\nBraun and his team were relocated to Fort Bliss, Texas, and later to the Redstone Arsenal in\nHuntsville, Alabama. Simultaneously, the Soviet Union executed Operation Osoaviakhim,\nsecuring the services of numerous German technicians and vast quantities of V-2\ndocumentation. Sergei Korolev, who would later become the Chief Designer of the Soviet\nspace program, was tasked with reverse-engineering the V-2, resulting in the creation of the\nR-1 missile. The parallel development of these technologies establish

In [13]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

def hybrid_search(query: str, k: int = 5, alpha: float = 0.5):
    bm25_results = sparse_retrieval(query, k=k*2)
    bm25_texts = [doc.page_content for doc in bm25_results]
    bm25_scores = np.array([doc.metadata.get('score', 1.0) for doc in bm25_results])
    
    dense_results = dense_retrieval(query, k=k*2)
    dense_texts = [doc.page_content for doc in dense_results]
    dense_scores = np.array([doc.metadata.get('score', 1.0) for doc in dense_results])
    
    scaler = MinMaxScaler()
    bm25_scores_norm = scaler.fit_transform(bm25_scores.reshape(-1, 1)).flatten()
    dense_scores_norm = scaler.fit_transform(dense_scores.reshape(-1, 1)).flatten()
    
    hybrid_dict = {}
    for text, score in zip(bm25_texts, bm25_scores_norm):
        hybrid_dict[text] = alpha * 0 + (1 - alpha) * score  # only BM25 for now
    for text, score in zip(dense_texts, dense_scores_norm):
        if text in hybrid_dict:
            hybrid_dict[text] = alpha * score + (1 - alpha) * hybrid_dict[text]
        else:
            hybrid_dict[text] = alpha * score
    
    # Sort and return top-k
    sorted_results = sorted(hybrid_dict.items(), key=lambda x: x[1], reverse=True)
    return [text for text, score in sorted_results[:k]]

hybrid_results = hybrid_search("Operation Paperclip", k=5, alpha=0.5)
hybrid_results

['The United States initiated Operation Paperclip, a strategic military program designed to\nrecruit prominent German scientists. Foremost among these was Wernher von Braun,\nwhose expertise in orbital mechanics and propulsion was unparalleled at the time. Von\nBraun and his team were relocated to Fort Bliss, Texas, and later to the Redstone Arsenal in\nHuntsville, Alabama. Simultaneously, the Soviet Union executed Operation Osoaviakhim,\nsecuring the services of numerous German technicians and vast quantities of V-2\ndocumentation. Sergei Korolev, who would later become the Chief Designer of the Soviet\nspace program, was tasked with reverse-engineering the V-2, resulting in the creation of the\nR-1 missile. The parallel development of these technologies established a bipolar\ntechnological hegemony.',
 "(Note: Page numbers are simulated for index formatting purposes)\nAgencies and Organizations\nAEB (Brazilian Space Agency) ... Page 112\nASI (Italian Space Agency) ... Page 68\nBlue O

In [16]:
context = "\n\n".join(hybrid_results)

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemma-3-27b-it"
)
model.invoke({
    ""
})

AIMessage(content="Hi there! 👋 \n\nHow can I help you today? Just let me know what you're thinking, or if you just wanted to say hi, that's great too! \n\nI can:\n\n* **Answer questions:** About pretty much anything!\n* **Generate text:** Like stories, poems, code, scripts, musical pieces, email, letters, etc.\n* **Translate languages.**\n* **Summarize text.**\n* **Help you brainstorm.**\n* **Just chat!**", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemma-3-27b-it', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c9f4a-2bdd-72f1-ae6a-3d1f8920fe57-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 0, 'total_tokens': 2, 'input_token_details': {'cache_read': 0}})

# Notebook Structure and Steps
---
1. **Project Introduction and Problem Statement**
   - Briefly describe the motivation and goals of hybrid search.

2. **Import Required Libraries**
   - List and import all necessary packages (BM25, Chroma, Gemini embeddings, Gemma LLM, etc.).

3. **Data Loading and Preprocessing**
   - Load documents (PDF/text).
   - Chunk documents recursively for indexing.

4. **Document Indexing**
   - Build BM25 index for lexical search.
   - Generate dense embeddings using Gemini and store in Chroma.

5. **Search Pipeline Implementation**
   - Implement BM25-based retrieval.
   - Implement dense retrieval using embeddings.
   - Normalize scores and combine results (weighted ensemble).

6. **Evaluation and Metrics**
   - Define and compute Precision@k, Recall@k, etc.
   - Compare BM25-only, dense-only, and hybrid retrieval.

7. **LLM Response Generation**
   - Use Gemma LLM to generate answers from top-ranked documents.

8. **Analysis and Discussion**
   - Discuss results, hybrid effectiveness, and lessons learned.

9. **Conclusion and Future Work**
   - Summarize findings and suggest improvements.

In [ ]:
# 1. Project Introduction and Problem Statement
# - Describe the motivation and goals of hybrid search.
# - Explain the limitations of BM25 and dense retrieval.
# - State the objectives and expected outcomes.

In [ ]:
# 2. Import Required Libraries
# - Import BM25, Chroma, Gemini embeddings, Gemma LLM, pdfplumber, langchain, etc.
# - Set up environment variables if needed.

In [ ]:
# 3. Data Loading and Preprocessing
# - Load documents (PDF/text) using pdfplumber.
# - Chunk documents recursively for indexing (header-based splitter).

In [ ]:
# 4. Document Indexing
# - Build BM25 index for lexical search.
# - Generate dense embeddings using Gemini and store in Chroma.

In [ ]:
# 5. Search Pipeline Implementation
# - Implement BM25-based retrieval.
# - Implement dense retrieval using embeddings.
# - Normalize scores and combine results (weighted ensemble).

In [ ]:
# 6. Evaluation and Metrics
# - Define and compute Precision@k, Recall@k, etc.
# - Compare BM25-only, dense-only, and hybrid retrieval.

In [ ]:
# 7. LLM Response Generation
# - Use Gemma LLM to generate answers from top-ranked documents.

In [ ]:
# 8. Analysis and Discussion
# - Discuss results, hybrid effectiveness, and lessons learned.

In [ ]:
# 9. Conclusion and Future Work
# - Summarize findings and suggest improvements.